# Full Anti-DPO Experiment: LoRA vs LR-LoRA

This Colab runs an end-to-end comparison on a T4 GPU. Both modes train the same inverted-preference objective: the source `rejected` response is the target, and long source `chosen` responses receive an extra bounded loss multiplier.

The baseline is standard LoRA. LR-LoRA is a genuine second training run: every selected Qwen projection is replaced by a learnable-sinc adapter `phi(BA)` and its stable rank is measured before and after training.

In [ ]:
# Select Runtime > Change runtime type > T4 GPU before running this cell.
REPO_URL = 'https://github.com/moliksq/Mauvais.git'
REPO_DIR = '/content/Mauvais'
!rm -rf $REPO_DIR
!git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR/anti_dpo_experiment
# Colab ships torchao 0.10. PEFT >= 0.14 rejects it; this experiment does not use torchao.
!pip -q uninstall -y torchao || true
!pip -q install --no-cache-dir -U 'transformers>=4.51,<5' 'trl>=0.16,<0.20' 'peft>=0.14' 'accelerate>=1.3' 'datasets>=3.0' 'matplotlib>=3.8'
!python -c "import torch, peft, trl; print('torch=', torch.__version__, 'peft=', peft.__version__, 'trl=', trl.__version__)"
!nvidia-smi

In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd()
assert torch.cuda.is_available(), 'Enable a Colab GPU and restart from the setup cell.'
assert (ROOT / 'data' / 'russian_qa' / 'dataset_dict.json').is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (MiB):', torch.cuda.get_device_properties(0).total_memory // 2**20)
print('Dataset:', ROOT / 'data' / 'russian_qa')

In [ ]:
# 200 optimizer steps is a comparable, affordable experiment on a free T4.
# Change MAX_STEPS to -1 to run the full one-epoch dataset pass.
MAX_STEPS = 200
COMMON = {
    'dataset_path': 'data/russian_qa', 'max_steps': MAX_STEPS,
    'batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 640, 'max_prompt_length': 256,
    'learning_rate': 5e-6, 'beta': .1, 'lora_r': 16, 'lora_alpha': 32,
    'lr_lora_basis': 8, 'eval_steps': 25, 'logging_steps': 5,
    'sample_count': 8, 'max_new_tokens': 96, 'precision': 'fp16',
}
COMMON

In [ ]:
import os, subprocess, sys
from pathlib import Path

def run_logged(adapter_type):
    output_dir = Path('outputs') / adapter_type
    command = [sys.executable, '-u', 'scripts/train.py', '--adapter_type', adapter_type, '--output_dir', str(output_dir)]
    for key, value in COMMON.items():
        command.extend([f'--{key}', str(value)])
    print('\n' + '=' * 88)
    print('Running:', ' '.join(command))
    print('=' * 88, flush=True)
    environment = os.environ | {'PYTHONUNBUFFERED': '1'}
    process = subprocess.Popen(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, env=environment)
    streamed_output = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        streamed_output.append(line)
    returncode = process.wait()
    log_path = output_dir / 'run.log'
    if log_path.exists():
        print('\nLast persistent log lines:', flush=True)
        print('\n'.join(log_path.read_text(encoding='utf-8').splitlines()[-30:]), flush=True)
    if returncode:
        failure = output_dir / 'failure.txt'
        details = failure.read_text(encoding='utf-8') if failure.exists() else ''.join(streamed_output)
        raise RuntimeError(f'{adapter_type} failed. Full traceback:\n{details}')
    return output_dir

lora_dir = run_logged('lora')

In [ ]:
# LR-LoRA is a separate optimization run. It has a frozen reference model and trainable learnable-sinc adapters.
torch.cuda.empty_cache()
lr_lora_dir = run_logged('lr_lora')

In [ ]:
# Build evaluation comparison and LR-LoRA stable-rank plot.
command = [sys.executable, 'scripts/analyze_experiment.py', '--lora_dir', str(lora_dir), '--lr_lora_dir', str(lr_lora_dir), '--output_dir', 'outputs/comparison']
completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode:
    raise RuntimeError(completed.stdout)
from IPython.display import Image, display
display(Image(filename='outputs/comparison/anti_dpo_comparison.png'))
display(Image(filename='outputs/comparison/lr_lora_stable_rank.png'))

In [ ]:
import json
from pathlib import Path

comparison = json.loads(Path('outputs/comparison/comparison.json').read_text(encoding='utf-8'))
comparison

In [ ]:
# Inspect model behaviour on identical prompts before and after each run.
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]
for run_name, run_dir in [('LoRA', lora_dir), ('LR-LoRA', lr_lora_dir)]:
    before, after = read_jsonl(run_dir / 'samples_before.jsonl'), read_jsonl(run_dir / 'samples_after.jsonl')
    print('\n' + '=' * 88 + f'\n{run_name}\n' + '=' * 88)
    for old, new in zip(before[:3], after[:3]):
        print('PROMPT:', old['prompt'])
        print('BEFORE:', old['generated'])
        print('AFTER :', new['generated'])
        print('ANTI TARGET:', old['target_original_rejected'])
        print()

In [ ]:
# Download all logs, checkpoints, samples and plots from the Colab session.
!tar -czf anti_dpo_experiment_outputs.tar.gz outputs
from google.colab import files
files.download('anti_dpo_experiment_outputs.tar.gz')